In [1]:
import argparse
import json
import tensorboard
import tensorboardX
import os
import argparse
import json
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim 
import nni
from nni.nas.nn.pytorch import ModelSpace, LayerChoice, MutableConv2d, MutableBatchNorm2d, MutableReLU,MutableLinear
from nni.nas.experiment.config import NasExperimentConfig
from pytorch_lightning import Trainer
from nni.nas.evaluator.pytorch import Lightning, ClassificationModule, Trainer
from nni.nas.experiment import NasExperiment
from nni.nas.space import model_context
from nni.nas.hub.pytorch import DARTS
from nni.nas.strategy import DARTS as DartsStrategy
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchvision import transforms
from torchvision.datasets import CIFAR10
from nni.nas.experiment import NasExperiment
from nni.nas.evaluator import FunctionalEvaluator
from nni.nas.evaluator import FunctionalEvaluator
import nni.nas.strategy as strategy
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from nni.experiment.config import utils, ExperimentConfig
#from ops import AvgPool,DilConv,SepConv
import genotypes
from pytorch_lightning.callbacks import ModelCheckpoint
torch.set_float32_matmul_precision('medium')
from tqdm import tqdm
from nni.nas.nn.pytorch import LayerChoice, ModelSpace,ValueChoice
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from pytorch_lightning import LightningModule, Trainer
from torchvision import datasets, transforms
from nni.nas.evaluator.pytorch import Classification

In [2]:
def cutout_transform(img, length: int = 16):
    h, w = img.size(1), img.size(2)
    mask = np.ones((h, w), np.float32)
    y = np.random.randint(h)
    x = np.random.randint(w)

    y1 = np.clip(y - length // 2, 0, h)
    y2 = np.clip(y + length // 2, 0, h)
    x1 = np.clip(x - length // 2, 0, w)
    x2 = np.clip(x + length // 2, 0, w)

    mask[y1: y2, x1: x2] = 0.
    mask = torch.from_numpy(mask)
    mask = mask.expand_as(img)
    img *= mask
    return img


In [3]:
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, SubsetRandomSampler
import nni
import numpy as np

def get_cifar10_dataset(train: bool = True, cutout: bool = False):
    if train:
        transform_list = [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(), 
        ]
        if cutout:
            transform_list.append(cutout_transform)
        transform = transforms.Compose(transform_list)
    else:
        transform = transforms.Compose([
            transforms.ToTensor(), 
        ])

    dataset = nni.trace(CIFAR10)(root='./data', train=train, download=True, transform=transform)
    
    return dataset

batch_size = 128
train_data = get_cifar10_dataset()
test_data =get_cifar10_dataset(train=False)

train_loader = DataLoader(
    train_data, batch_size=batch_size,
    pin_memory=True, num_workers=6, persistent_workers=True,shuffle=True
)

valid_loader = DataLoader(
    test_data, batch_size=batch_size,
    pin_memory=True, num_workers=6, persistent_workers=True
)


Files already downloaded and verified
Files already downloaded and verified


In [4]:
@nni.trace
class DartsClassificationModule(ClassificationModule):
    def __init__( self,learning_rate: float = 0.001,weight_decay: float = 0.,auxiliary_loss_weight: float = 0.4,max_epochs: int = 600):
        super().__init__(learning_rate=learning_rate, weight_decay=weight_decay, export_onnx=False,num_classes=10)        
        self.auxiliary_loss_weight = auxiliary_loss_weight
        self.max_epochs = max_epochs
        self.learning_rate = learning_rate


    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.parameters(), lr=self.learning_rate, momentum=0.9, weight_decay=0.)
        return {
            'optimizer': optimizer,
            'lr_scheduler': torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
        }

    def training_step(self, batch, batch_idx):
        """Training step, customized with auxiliary loss."""
        x, y = batch
        if self.auxiliary_loss_weight:
            y_hat, y_aux = self(x)
            loss_main = self.criterion(y_hat, y)
            loss_aux = self.criterion(y_aux, y)
            self.log('train_loss_main', loss_main)
            self.log('train_loss_aux', loss_aux)
            loss = loss_main + self.auxiliary_loss_weight * loss_aux
        else:
            y_hat = self(x)
            loss = self.criterion(y_hat, y)
        self.log('train_loss', loss, prog_bar=True)
        for name, metric in self.metrics.items():
            self.log('train_' + name, metric(y_hat, y), prog_bar=True)
        return loss

    def on_train_epoch_start(self):
        # Set drop path probability before every epoch. This has no effect if drop path is not enabled in model.
        #self.model.set_drop_path_prob(self.model.drop_path_prob * self.current_epoch / self.max_epochs)

        # Logging learning rate at the beginning of every epoch
        self.log('lr', self.trainer.optimizers[0].param_groups[0]['lr'])


In [5]:

class CustomDARTSSpace(ModelSpace):
    def __init__(self, input_channels=3, channels=64, num_classes=10, layers=5,verbose =0, drop_path_prob = 0.1):
        super(CustomDARTSSpace, self).__init__()

        #________________________________________________________________________________________________________________________
        #Inizialization
        self.layers = nn.ModuleList()
        self.drop_path_prob = drop_path_prob
        self.verbose = verbose


        #________________________________________________________________________________________________________________________
        #Channel choices
        layer0_out = 16
        layer1_out = nni.choice('layer1_out_channels', [8,16,32,48,64,96,128,144])
        layer2_out= nni.choice('layer2_out_channels', [8,16,32,48,64,96,128,144])
        layer3_out= nni.choice('layer3_out_channels', [8,16,32,48,64,96,128,144])
        layer4_out = nni.choice('layer4_out_channels', [8,16,32,48,64,96,128,144])
        layer5_out= nni.choice('layer5_out_channels', [8,16,32,48,64,96,128,144])
        layer6_out= nni.choice('layer6_out_channels', [8,16,32,48,64,96,128,144])
        layer7_out= 22
        
        #________________________________________________________________________________________________________________________
        #Layer 0
        self.preliminary_layer = nn.Conv2d(3, layer0_out, kernel_size=3, padding=0, bias=False)
        self.layer0_bn = torch.nn.BatchNorm2d(layer0_out)
        self.layer0_relu = torch.nn.ReLU(inplace=True)
        
        #________________________________________________________________________________________________________________________
        #Layer 1
        layer1 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1 ),
                MutableConv2d(layer0_out, layer1_out, kernel_size=3),
                MutableBatchNorm2d(layer1_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer0_out, layer1_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer1_out),
                MutableReLU()
            )
        ], label='layer_1')
        self.layers.append(layer1)
        
        #________________________________________________________________________________________________________________________
        #Layer 2
        layer2 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer1_out, layer2_out, kernel_size=3),
                MutableBatchNorm2d(layer2_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer1_out, layer2_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer2_out),
                MutableReLU()
            )
        ], label='layer_2')
        self.layers.append(layer2)
        
        #________________________________________________________________________________________________________________________
        #Layer 3
        layer3 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer2_out, layer3_out, kernel_size=3),
                MutableBatchNorm2d(layer3_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer2_out, layer3_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer3_out),
                MutableReLU()
            )
        ], label='layer_3')
        self.layers.append(layer3)
                #________________________________________________________________________________________________________________________
        #Layer 4
        layer4 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer3_out, layer4_out, kernel_size=3),
                MutableBatchNorm2d(layer4_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer3_out, layer4_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer4_out),
                MutableReLU()
            )
        ], label='layer_4')
        self.layers.append(layer4)
                #________________________________________________________________________________________________________________________
        #Layer 5
        layer5 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer4_out, layer5_out, kernel_size=3),
                MutableBatchNorm2d(layer5_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer4_out, layer5_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer5_out),
                MutableReLU()
            )
        ], label='layer_5')
        self.layers.append(layer5)
                #________________________________________________________________________________________________________________________
        #Layer 6
        layer6= LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer5_out, layer6_out, kernel_size=3),
                MutableBatchNorm2d(layer6_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer5_out, layer6_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer6_out),
                MutableReLU()
            )
        ], label='layer_6')
        self.layers.append(layer6)
                #________________________________________________________________________________________________________________________
        #Layer 7
        layer7 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer6_out, layer7_out, kernel_size=3),
                MutableBatchNorm2d(layer7_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer6_out, layer7_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer7_out),
                MutableReLU()
            )
        ], label='layer_7')
        self.layers.append(layer7)
        

        
        #________________________________________________________________________________________________________________________
        #Linear
        self.pool = nn.AdaptiveAvgPool2d((3, 3))
        feature1 = nni.choice('feature1', [8,16,32,48,64,96,128,144])
        feature2 = nni.choice('feature2', [8,16,32,48,64,96,128,144])
        feature3 = nni.choice('feature3', [8,16,32,48,64,96,128,144])
        self.fc1 = MutableLinear(198, feature1) 
        self.fc2 = MutableLinear(feature1, feature2) 
        self.fc3 = MutableLinear(feature2, feature3)  
        self.relu = nn.ReLU()
        self.classifier = MutableLinear(feature3, num_classes)

    def forward(self, x):
        #________________________________________________________________________________________________________________________
        #Layer 0
        x = self.preliminary_layer(x)
        X = self.layer0_bn(x)
        x = self.layer0_relu(x)
        if self.verbose == 1 :
            print(f'After preliminary layer: {x.shape}')
        #________________________________________________________________________________________________________________________
        #Layer 1 to n
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if self.verbose == 1 :
                print(f'After layer {i+1}: {x.shape}')
            if i == 1 or i == 3 or i == 6:
                x = nn.AvgPool2d(kernel_size=2, stride=2)(x)
                if self.verbose == 1 :
                    print(f'After avg pooling: {x.shape}')
        
        #________________________________________________________________________________________________________________________
        #Adaprive pool
        x =  self.pool(x)
        if self.verbose == 1 :
            print(f'After adaptive pooling: {x.shape}')

        #________________________________________________________________________________________________________________________
        #Flatten
        x = torch.flatten(x, 1)
        if self.verbose == 1 :
            print(f'After flattening: {x.shape}')
        #________________________________________________________________________________________________________________________
        
        x = self.fc1(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc1: {x.shape}')
        x = self.fc2(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc2: {x.shape}')
        x = self.fc3(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc3: {x.shape}')
        #________________________________________________________________________________________________________________________
        #Classification 
        x = self.classifier(x)

        
        if self.verbose == 1 :
            print(f'After classifier: {x.shape}')
        #self.first_iter = False
        return x

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob
        for layer in self.layers:
            if hasattr(layer, 'set_drop_path_prob'):
                layer.set_drop_path_prob(drop_path_prob)


In [6]:
# Checkpoint 
checkpoint_callback = ModelCheckpoint(
    monitor='train_acc', 
    dirpath='./checkpoints',
    filename='best-checkpoint',
    save_top_k=1,
    mode='max'
    
)

In [7]:
from nni.nas.evaluator.pytorch import Lightning, Trainer

max_epochs = 200

evaluator = Lightning(
    DartsClassificationModule(1e-2, 0., 0., max_epochs),
    Trainer(
        accelerator="auto",
        callbacks=[checkpoint_callback],
        max_epochs=max_epochs
    ),
    train_dataloaders=train_loader,
    val_dataloaders=valid_loader
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [8]:
def get_next_experiment_name(experiment_working_directory: str, base_name: str = "PhotonicDARTS-v"):
    dirs = os.listdir(experiment_working_directory)

    pattern = re.compile(rf'{base_name}(\d+)')

    highest_i = 0
    for d in dirs:
        match = pattern.match(d)
        if match:
            i_value = int(match.group(1))
            if i_value > highest_i:
                highest_i = i_value
    return f"{base_name}{highest_i + 1}"

In [9]:
strategy = DartsStrategy(gradient_clip_val=0.)
def search(log_dir: str, batch_size: int = 128):

    # Define model search space
    model_space = CustomDARTSSpace(input_channels=3, channels=64, num_classes=10, layers=7, verbose=1)
    model_space.set_drop_path_prob(0.)

    # Run NAS experiment
    exp_config = NasExperimentConfig.default(model_space, evaluator, strategy)
    exp_config.experiment_working_directory = "./DartsCheckpoints"
    exp_config.experiment_name = "Darts_search"
    exp_config.trial_concurrency = 1
    experiment = NasExperiment(model_space, evaluator, strategy, config = exp_config)
    experiment.run()

    return experiment


In [ ]:
experiment_results = search("./",32)

[2025-03-10 16:44:49] Config is not provided. Will try to infer.
[2025-03-10 16:44:49] Strategy is found to be a one-shot strategy. Setting execution engine to "sequential" and format to "raw".
[2025-03-10 16:44:49] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-10 16:44:49] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-10 16:44:49] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-10 16:44:49] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-10 16:44:49] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-10 16:44:49] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-10 16:44:49] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-10 16:44:49] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-03-10 16

C:\Users\senti\anaconda3\envs\NNI_NAS_local\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:653: Checkpoint directory C:\Users\senti\Documents\GitHub\PhotonicNas\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name            | Type                      | Params
--------------------------------------------------------------
0 | training_module | DartsClassificationModule | 2.0 M 
--------------------------------------------------------------
2.0 M     Trainable params
0         Non-trainable params
2.0 M     Total params
8.170     Total estimated model params size (MB)


Epoch 0:   0%|          | 0/391 [00:00<?, ?it/s] After preliminary layer: torch.Size([128, 16, 30, 30])
After layer 1: torch.Size([128, 144, 29, 29])


C:\Users\senti\anaconda3\envs\NNI_NAS_local\Lib\site-packages\nni\nas\oneshot\pytorch\supermodule\operation.py:465: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ..\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return F.conv2d(inputs, weight, bias, stride_, cast('int | tuple', padding), dilation_, groups)


After layer 2: torch.Size([128, 144, 28, 28])
After avg pooling: torch.Size([128, 144, 14, 14])
After layer 3: torch.Size([128, 144, 13, 13])
After layer 4: torch.Size([128, 144, 12, 12])
After avg pooling: torch.Size([128, 144, 6, 6])
After layer 5: torch.Size([128, 144, 5, 5])
After layer 6: torch.Size([128, 144, 4, 4])
After layer 7: torch.Size([128, 22, 3, 3])
After avg pooling: torch.Size([128, 22, 1, 1])
After adaptive pooling: torch.Size([128, 22, 3, 3])
After flattening: torch.Size([128, 198])
After fc1: torch.Size([128, 144])
After fc2: torch.Size([128, 144])
After fc3: torch.Size([128, 144])
After classifier: torch.Size([128, 10])
After preliminary layer: torch.Size([128, 16, 30, 30])
After layer 1: torch.Size([128, 144, 29, 29])
After layer 2: torch.Size([128, 144, 28, 28])
After avg pooling: torch.Size([128, 144, 14, 14])
After layer 3: torch.Size([128, 144, 13, 13])
After layer 4: torch.Size([128, 144, 12, 12])
After avg pooling: torch.Size([128, 144, 6, 6])
After layer 5: